<div style="background: white;
            padding: 30px 35px;
            border-radius: 8px;
            border-left: 6px solid #e94560;
            font-family: 'Segoe UI', sans-serif;
            color: #1a1a1a;">

<h1 style="margin-bottom: 5px;">NB03: Streamlit Dashboard — Interactive Portfolio Risk Tool</h1>

<p style="font-weight: bold; margin-top: 0;">Personal Project · Portfolio Risk Dashboard · 2026</p>

<ul>
    <li>📅 <strong>Date:</strong> 12th April 2026</li>
    <li>🎯 <strong>Purpose:</strong> Build an interactive Streamlit dashboard that lets users adjust portfolio weights and confidence levels dynamically and see VaR, ES and drawdown update in real time</li>
</ul>

<p><strong>Input:</strong></p>
<ul>
    <li><code>../data/raw/prices.csv</code> — cleaned daily adjusted closing prices from NB01</li>
    <li><code>../data/processed/returns.csv</code> — daily log returns from NB01</li>
</ul>

<p><strong>Output:</strong></p>
<ul>
    <li><code>../app/dashboard.py</code> — standalone Streamlit application</li>
</ul>

<p><strong>Key Techniques Used:</strong></p>
<ul>
    <li>Streamlit widgets — sliders, selectboxes, metrics</li>
    <li>Dynamic portfolio construction from user-defined weights</li>
    <li>Real-time VaR and ES recalculation on input change</li>
    <li>Interactive Plotly charts for price history, drawdown, and return distribution</li>
</ul>

<p><strong>Workflow:</strong> Design Layout → Build Sidebar Controls → Compute Risk Metrics → Render Charts → Run App</p>

</div>

## ⚙️ Section 1: Install & Import Dependencies

### 📍 Step 1: Install Required Libraries

In [1]:
# Run pip install from inside the notebook — same as typing in terminal but automated
# check=True means Python will raise an error if the installation fails
# "--quiet" suppresses the verbose installation logs to keep output clean
import subprocess
subprocess.run(["pip", "install", "streamlit", "plotly", "--quiet"], check=True)
print("✅ Streamlit and Plotly installed successfully.")

✅ Streamlit and Plotly installed successfully.


### 📍 Step 2: Verify Imports Work

In [2]:
import pandas as pd
import numpy as np
import streamlit as st        # the dashboard framework
import plotly.graph_objects as go  # for building custom interactive charts
import plotly.express as px        # for quick interactive charts with less code
from scipy import stats            # for normal distribution calculations
import os                          # for file path operations
import warnings

warnings.filterwarnings("ignore")
print("✅ All libraries imported successfully.")

✅ All libraries imported successfully.


## 🗂️ Section 2: Create App Folder

### 📍 Set Up App Directory Structure

We create a dedicated `/app` folder at the project root to house the Streamlit
dashboard script. Keeping it separate from `/notebooks` ensures the app is
self-contained and can be run independently.

In [3]:
# Define the path to the app folder — one level up from notebooks/, then into app/
app_dir = "../app"

# Create the folder if it doesn't already exist
# exist_ok=True means no error is thrown if the folder already exists — safe to re-run
os.makedirs(app_dir, exist_ok=True)

print(f"✅ App directory ready: {app_dir}")

✅ App directory ready: ../app


## 🛠️ Section 3: Write the Dashboard Script

We write the entire Streamlit app to a single Python file using Python's
`%%writefile` magic command. This means we can write and edit the app
directly from the notebook without leaving Jupyter.

The dashboard has four main components:
- **Sidebar** — sliders for portfolio weights and a confidence level selector
- **Metrics row** — VaR, ES and max drawdown displayed as headline numbers
- **Charts** — normalised price history, return distribution with VaR marked, and drawdown chart
- **Summary table** — all three VaR methods compared side by side

In [8]:
%%writefile ../app/dashboard.py

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from scipy import stats
import os

# Page Config 
# Sets the browser tab title, favicon, and uses wide layout to fill the full screen
st.set_page_config(
    page_title="Portfolio Risk Dashboard",
    page_icon="📊",
    layout="wide"
)

# Custom CSS 
# Injects custom styling into the dashboard
# unsafe_allow_html=True is required to render raw HTML/CSS in Streamlit
st.markdown("""
    <style>
    .main { background-color: #0f0f0f; }
    .metric-card {
        background: #1a1a2e;
        border-left: 4px solid #e94560;
        padding: 15px 20px;
        border-radius: 8px;
        margin: 5px 0;
    }
    </style>
""", unsafe_allow_html=True)

# Title 
# Display the dashboard title and subtitle at the top of the page
st.title("📊 Portfolio Risk Dashboard")
st.markdown("*Interactive VaR, Expected Shortfall & Drawdown Analysis — 2016–2025*")
st.markdown("---")

# Load Data 
# os.path.abspath(__file__) gets the exact location of dashboard.py on disk
# os.path.dirname strips the filename to get just the folder path
# This ensures file paths work regardless of which directory Streamlit is launched from
BASE_DIR = os.path.dirname(os.path.abspath(__file__))

# @st.cache_data tells Streamlit to load the CSVs once and store them in memory
# Without this, files reload from disk every time a slider moves — very slow
@st.cache_data
def load_data():
    prices  = pd.read_csv(os.path.join(BASE_DIR, "../data/raw/prices.csv"),
                          index_col="Date", parse_dates=True)
    returns = pd.read_csv(os.path.join(BASE_DIR, "../data/processed/returns.csv"),
                          index_col="Date", parse_dates=True)
    return prices, returns

prices, returns = load_data()

# Define asset names for chart labels
TICKERS = {
    "SPY": "S&P 500 ETF",
    "TLT": "20Y US Treasury ETF",
    "GLD": "Gold ETF",
    "QQQ": "Nasdaq 100 ETF",
    "EEM": "Emerging Markets ETF"
}

# Sidebar 
# The sidebar holds all user controls — weights and confidence level
st.sidebar.title("⚙️ Portfolio Controls")
st.sidebar.markdown("Adjust weights and settings below. Charts update automatically.")
st.sidebar.markdown("---")

st.sidebar.subheader("📊 Asset Weights")
st.sidebar.markdown("*Weights must sum to 1.0*")

# Each slider lets the user set a weight between 0.0 and 1.0 in steps of 0.05
# Arguments: (label, min, max, default, step)
w_spy = st.sidebar.slider("SPY — S&P 500",       0.0, 1.0, 0.35, 0.05)
w_tlt = st.sidebar.slider("TLT — US Treasuries", 0.0, 1.0, 0.25, 0.05)
w_gld = st.sidebar.slider("GLD — Gold",          0.0, 1.0, 0.15, 0.05)
w_qqq = st.sidebar.slider("QQQ — Nasdaq 100",    0.0, 1.0, 0.15, 0.05)
w_eem = st.sidebar.slider("EEM — Emerging Mkts", 0.0, 1.0, 0.10, 0.05)

# Combine raw slider values into a numpy array for matrix multiplication later
weights_raw = np.array([w_spy, w_tlt, w_gld, w_qqq, w_eem])
weight_sum  = weights_raw.sum()

# Normalise weights to always sum to 1
# e.g. if user sets all sliders to 0.5, raw sum = 2.5, each weight becomes 0.5/2.5 = 0.2
weights = weights_raw / weight_sum if weight_sum > 0 else weights_raw

st.sidebar.markdown("---")
st.sidebar.subheader("🎯 Risk Settings")

# Dropdown to select confidence level — defaults to 95% (index=1)
# format_func converts 0.95 to "95%" for display
confidence = st.sidebar.selectbox(
    "Confidence Level",
    options=[0.90, 0.95, 0.99],
    index=1,
    format_func=lambda x: f"{int(x*100)}%"
)

# Show the normalised weights in the sidebar so the user can see the actual allocations
st.sidebar.markdown("---")
st.sidebar.markdown(f"**Normalised Weights:**")
for ticker, w in zip(TICKERS.keys(), weights):
    st.sidebar.markdown(f"- {ticker}: {w:.1%}")
st.sidebar.markdown(f"**Total: {weights.sum():.2f}**")

# Compute Portfolio Returns
# Dot product multiplies each asset's daily return by its weight and sums across all 5 assets
# This runs every time a slider moves — Streamlit reruns the entire script automatically
portfolio_returns = returns.dot(weights)

# Risk Calculations 

# Historical VaR — find the actual percentile from observed returns
hist_var = np.percentile(portfolio_returns, (1 - confidence) * 100)

# Expected Shortfall — filter to only the tail days (worse than VaR) and take their average
tail     = portfolio_returns[portfolio_returns <= hist_var]
hist_es  = tail.mean()

# Parametric VaR — calculate analytically using mean, volatility and z-score
mu        = portfolio_returns.mean()
sigma     = portfolio_returns.std()
z_scores  = {0.90: 1.282, 0.95: 1.645, 0.99: 2.326}
param_var = mu - z_scores[confidence] * sigma

# Monte Carlo VaR — simulate 10,000 random returns and find the percentile
np.random.seed(42) # fixed seed ensures identical results on every run
sim       = np.random.normal(mu, sigma, 10_000)
mc_var    = np.percentile(sim, (1 - confidence) * 100)

# Drawdown — measure how far portfolio has fallen from its previous peak at each point
cumulative  = (1 + portfolio_returns).cumprod()   # running portfolio value starting at $1
running_max = cumulative.cummax()                  # highest value reached up to each date
drawdown    = (cumulative - running_max) / running_max  # % below the peak
max_dd      = drawdown.min()                       # single worst drawdown point
max_dd_date = drawdown.idxmin()                    # date of worst drawdown

# Annualised statistics — multiply daily figures by trading days in a year
ann_return = mu * 252                  # 252 trading days per year
ann_vol    = sigma * np.sqrt(252)      # volatility scales by square root of time
sharpe     = ann_return / ann_vol if ann_vol > 0 else 0  # return per unit of risk

# Metrics Row 
# Display five headline risk numbers side by side across the top of the dashboard
st.subheader(f"📐 Risk Metrics at {int(confidence*100)}% Confidence")

# st.columns(5) creates 5 equal-width boxes side by side
col1, col2, col3, col4, col5 = st.columns(5)

# Each metric shows a label, a headline value, and a subtitle
# delta_color="off" prevents Streamlit from colouring the subtitle green or red
col1.metric(
    label=f"Historical VaR ({int(confidence*100)}%)",
    value=f"{abs(hist_var)*100:.2f}%",
    delta="daily loss threshold",
    delta_color="off"
)
col2.metric(
    label=f"Parametric VaR ({int(confidence*100)}%)",
    value=f"{abs(param_var)*100:.2f}%",
    delta="normal distribution",
    delta_color="off"
)
col3.metric(
    label=f"Monte Carlo VaR ({int(confidence*100)}%)",
    value=f"{abs(mc_var)*100:.2f}%",
    delta="10,000 simulations",
    delta_color="off"
)
col4.metric(
    label=f"Expected Shortfall ({int(confidence*100)}%)",
    value=f"{abs(hist_es)*100:.2f}%",
    delta="avg loss beyond VaR",
    delta_color="off"
)
col5.metric(
    label="Max Drawdown",
    value=f"{abs(max_dd)*100:.2f}%",
    delta=f"on {max_dd_date.strftime('%d %b %Y')}",
    delta_color="off"
)

st.markdown("---")

# Portfolio Stats Row 
# Second row of metrics showing broader portfolio statistics
st.subheader("📊 Portfolio Statistics")
col1, col2, col3, col4 = st.columns(4)

col1.metric("Annualised Return",   f"{ann_return*100:.2f}%")
col2.metric("Annualised Volatility", f"{ann_vol*100:.2f}%")
col3.metric("Sharpe Ratio",        f"{sharpe:.2f}")
col4.metric("Worst Single Day",    f"{portfolio_returns.min()*100:.2f}%")

st.markdown("---")

# Charts 
# Four tabs - each contains one interactive Plotly chart
st.subheader("📈 Charts")
tab1, tab2, tab3, tab4 = st.tabs([
    "📈 Price History",
    "📉 Return Distribution",
    "📉 Drawdown",
    "📊 VaR Comparison"
])

# Tab 1: Normalised Price History 
with tab1:
    st.markdown("**Normalised price performance (Base = 100) — 2016 to 2025**")

    # Divide all prices by the first row to set each asset to 100 at the start
    normalised = (prices / prices.iloc[0]) * 100

    fig1 = go.Figure()

    # Assign a distinct colour to each asset
    colors_map = {
        "SPY": "#e94560", "TLT": "#a8dadc",
        "GLD": "#f4a261", "QQQ": "#ffffff", "EEM": "#c77dff"
    }

    # Add one line per asset to the chart
    for ticker, label in TICKERS.items():
        fig1.add_trace(go.Scatter(
            x=normalised.index,
            y=normalised[ticker],
            name=f"{ticker} — {label}",
            line=dict(color=colors_map[ticker], width=1.5)
        ))
    
    # Shade the COVID crash period in red - semi-transparent so lines are visible
    fig1.add_vrect(x0="2020-02-01", x1="2020-04-30",
                   fillcolor="red", opacity=0.1,
                   annotation_text="COVID", annotation_position="top left")
    
    # Shade the 2022 rate hike period in orange
    fig1.add_vrect(x0="2022-01-01", x1="2022-12-31",
                   fillcolor="orange", opacity=0.1,
                   annotation_text="Rate Hikes", annotation_position="top left")
    
    fig1.update_layout(
        template="plotly_dark",
        title="Normalised Price Performance (Base = 100)",
        xaxis_title="Date",
        yaxis_title="Normalised Price",
        height=500,
        legend=dict(orientation="v", x=0.01, y=0.99)
    )

    # use container_width=True makes the chart fill the full tab width
    st.plotly_chart(fig1, use_container_width=True)

# Tab 2: Return Distribution 
with tab2:
    st.markdown(f"**Portfolio daily return distribution with VaR thresholds marked**")
    
    fig2 = go.Figure()
    
    # Plot the histogram of all daily portfolio returns
    fig2.add_trace(go.Histogram(
        x=portfolio_returns,
        nbinsx=100,
        name="Daily Returns",
        marker_color="#e94560",
        opacity=0.8
    ))

    # Draw vertical dashed lines at each risk threshold
    # Historical VaR - orange dashed line
    fig2.add_vline(x=hist_var,  line_dash="dash", line_color="orange",
                   annotation_text=f"Hist VaR {int(confidence*100)}%: {abs(hist_var)*100:.2f}%",
                   annotation_position="top left")
    
    # Parametric VaR - cyan dashed line
    fig2.add_vline(x=param_var, line_dash="dash", line_color="cyan",
                   annotation_text=f"Param VaR {int(confidence*100)}%: {abs(param_var)*100:.2f}%",
                   annotation_position="top right")
    
    # Expected Shortfall - white dotted line, furthest left (most extreme)
    fig2.add_vline(x=hist_es,   line_dash="dot",  line_color="white",
                   annotation_text=f"ES {int(confidence*100)}%: {abs(hist_es)*100:.2f}%",
                   annotation_position="bottom left")
    
    fig2.update_layout(
        template="plotly_dark",
        title="Portfolio Daily Log Return Distribution",
        xaxis_title="Daily Log Return",
        yaxis_title="Frequency",
        height=500
    )
    st.plotly_chart(fig2, use_container_width=True)

# Tab 3: Drawdown 
with tab3:
    st.markdown("**Portfolio value and drawdown from peak over the full sample period**")
    fig3 = go.Figure()

    # Top line - cumulative portfolio value plotted on the left y-axi
    fig3.add_trace(go.Scatter(
        x=cumulative.index,
        y=cumulative.values,
        name="Portfolio Value",
        line=dict(color="#e94560", width=1.5),
        yaxis="y1"
    ))

    # Shaded area - drawdown plotted on the right y-axis
    # fill="tozeroy" fills the area between the drawdown line and zero
    fig3.add_trace(go.Scatter(
        x=drawdown.index,
        y=drawdown.values,
        name="Drawdown",
        fill="tozeroy",
        line=dict(color="#f4a261", width=1),
        yaxis="y2"
    ))

    # Horizontal dashed line marking the maximum drawndown level
    fig3.add_hline(
        y=max_dd, line_dash="dash", line_color="cyan",
        annotation_text=f"Max DD: {max_dd*100:.2f}%",
        yref="y2"
    )
    fig3.update_layout(
        template="plotly_dark",
        title="Cumulative Portfolio Value & Drawdown",
        height=550,
        # Two y-axes - portfolio value on left, drawdown on right
        yaxis=dict(title="Portfolio Value ($)", side="left"),
        yaxis2=dict(title="Drawdown", side="right", overlaying="y"),
        legend=dict(x=0.01, y=0.99)
    )
    st.plotly_chart(fig3, use_container_width=True)

# Tab 4: VaR Comparison 
with tab4:
    st.markdown("**VaR estimates across all three methods and all confidence levels**")
    cls    = [0.90, 0.95, 0.99]
    labels = ["90%", "95%", "99%"]

    # Recalculate all methods across all three confidence levels for the comparison chart
    hist_vals  = [abs(np.percentile(portfolio_returns, (1-c)*100))*100 for c in cls]
    param_vals = [abs(mu - z_scores[c] * sigma)*100 for c in cls]
    mc_vals    = [abs(np.percentile(np.random.normal(mu, sigma, 10_000), (1-c)*100))*100 for c in cls]
    es_vals    = [abs(portfolio_returns[portfolio_returns <=
                  np.percentile(portfolio_returns, (1-c)*100)].mean())*100 for c in cls]

    fig4 = go.Figure()

    # Add one bar series per method - each gets its own colour
    fig4.add_trace(go.Bar(name="Historical VaR",  x=labels, y=hist_vals,  marker_color="#e94560"))
    fig4.add_trace(go.Bar(name="Parametric VaR",  x=labels, y=param_vals, marker_color="#a8dadc"))
    fig4.add_trace(go.Bar(name="Monte Carlo VaR", x=labels, y=mc_vals,    marker_color="#f4a261"))
    fig4.add_trace(go.Bar(name="Expected Shortfall", x=labels, y=es_vals, marker_color="#c77dff"))

    fig4.update_layout(
        template="plotly_dark",
        barmode="group",    # places bars side by side rather than stacked
        title="VaR & ES Comparison Across Methods and Confidence Levels",
        xaxis_title="Confidence Level",
        yaxis_title="Loss Threshold (%)",
        height=500,
        legend=dict(orientation="h", y=-0.2)
    )
    st.plotly_chart(fig4, use_container_width=True)

    # Summary table below the chart
    st.markdown("**📋 Summary Table**")
    summary = pd.DataFrame({
        "Historical VaR"    : [f"{v:.3f}%" for v in hist_vals],
        "Parametric VaR"    : [f"{v:.3f}%" for v in param_vals],
        "Monte Carlo VaR"   : [f"{v:.3f}%" for v in mc_vals],
        "Expected Shortfall": [f"{v:.3f}%" for v in es_vals]
    }, index=["90%", "95%", "99%"])
    summary.index.name = "Confidence Level"

    # st.dataframe renders an interactive scrollable table
    st.dataframe(summary, use_container_width=True)

# Footer 
st.markdown("---")
st.markdown(
    "*Portfolio Risk Dashboard · Built with Python, Streamlit & Plotly · 2026*"
)

Overwriting ../app/dashboard.py


## 🚀 Section 4: Run the Dashboard

### 📍 Launch Streamlit

Streamlit apps cannot run inside a Jupyter notebook cell directly — they need to be
launched from the terminal as a standalone process. I need to run the command below in my
Nuvolos terminal to start the dashboard. It will open in my browser automatically.

In [5]:
print("🚀 To launch the dashboard, run this command in my terminal:")
print()
print("   cd ../app && streamlit run dashboard.py")
print()
print("Then open the URL shown in the terminal (usually http://localhost:8501)")
print()
print("💡 Tip: Keep the terminal running while using the dashboard.")
print("   Press Ctrl+C in the terminal to stop it when done.")

🚀 To launch the dashboard, run this command in my terminal:

   cd ../app && streamlit run dashboard.py

Then open the URL shown in the terminal (usually http://localhost:8501)

💡 Tip: Keep the terminal running while using the dashboard.
   Press Ctrl+C in the terminal to stop it when done.


## 🪞 Personal Reflection Notes — NB03

**🎯 Key Decisions Made in This Notebook**

The central decision in this notebook was choosing Streamlit over alternative dashboard frameworks like Dash or Flask. Streamlit was the right choice for this project for three reasons. First, it is the fastest way to turn a Python script into an interactive web app — there is no HTML, CSS, or JavaScript required, which keeps the codebase clean and focused on the financial logic rather than web development. Second, it is widely used in data science and quantitative finance for internal tools and prototypes, so it is a genuinely transferable skill. Third, its reactive model — where the entire script reruns automatically whenever the user moves a slider — meant that all the VaR and ES calculations update in real time without any additional callback logic.

The decision to normalise portfolio weights automatically rather than enforcing a hard constraint was also deliberate. If a user sets all five sliders to 0.5, the raw weights sum to 2.5 — not 1.0. Rather than throwing an error, the dashboard divides each weight by the total, so the portfolio always sums to 100% regardless of what the user inputs. This makes the tool more robust and less frustrating to use.

The choice to use Plotly rather than Matplotlib for the dashboard charts was driven by interactivity. Plotly charts are natively interactive — users can hover over data points to see exact values, zoom into specific time periods, and toggle assets on and off by clicking the legend. For a risk dashboard, this is far more useful than a static Matplotlib image.

**📊 What the Dashboard Reveals**

The most instructive feature of the dashboard is how dramatically VaR and ES change when portfolio weights are adjusted. Moving weight from TLT and GLD into QQQ and EEM pushes all risk metrics sharply higher — the distribution widens, the VaR lines shift left, and the max drawdown deepens. This makes viscerally clear something that is easy to miss in the static NB02 analysis: the risk profile of a portfolio is not fixed, it is a direct consequence of allocation decisions. A risk manager with this tool can immediately see the cost of increasing equity exposure in terms of tail risk.

The gap between historical and parametric VaR becomes more visible at the 99% confidence level in the comparison chart. At 90% the bars are similar heights, but by 99% historical VaR and ES tower over the parametric and Monte Carlo estimates. This reinforces the fat tail finding from NB01 and NB02 in a way that is immediately intuitive — you can see the gap growing as the confidence level increases without needing to read a table of numbers.

The tabs structure was important for managing information density. Presenting all four charts on one page would have been overwhelming. Breaking them into Price History, Return Distribution, Drawdown and VaR Comparison lets the user focus on one dimension of risk at a time, which mirrors how a real risk report is structured.

**🔜 Possible Extensions**

There are several natural extensions to this dashboard that would make it more production-ready. Adding a date range selector would allow users to calculate VaR over specific sub-periods — for example, comparing VaR during the 2022 rate hike period versus the full sample. Adding a position size input (e.g. "I have £100,000 invested") would convert percentage VaR into a pound-value loss, which is far more intuitive for a real investor. Finally, implementing a GARCH volatility model to replace the constant volatility assumption in parametric and Monte Carlo VaR would make the estimates time-varying and more realistic — risk is not constant, it clusters during crises and falls during calm periods.